# #  Data Preprocessing with Scikit-Learn
#
# This section follows the Scikit-Learn preprocessing workflow:
#
# Import Data → Inspect Missing Values → Split X/y → Train/Test Split
# → Impute Missing Values → Encode Categorical Data → Fit Model
#
# The example uses the extended car-sales dataset.

In [ ]:
# Standard imports
import numpy as np
import pandas as pd

In [ ]:
# ============================================================
# 1. GET THE DATA READY
# ============================================================

# Import car sales data with missing values
# Keep the CSV inside a "data" folder in your repository.
car_sales_missing = pd.read_csv(
    "data/car-sales-extended-missing-data.csv"
)

# View the first 5 rows
car_sales_missing.head()

In [ ]:
# Check the shape of the dataset
print("Dataset shape:", car_sales_missing.shape)

# Check column names
print("\nColumns:")
print(car_sales_missing.columns.tolist())

In [ ]:
# Check missing values
print("Missing values in each column:")
print(car_sales_missing.isna().sum())

# ## 1.1 Missing Values
#
# Missing data can be handled mainly in two ways:
#
# 1. Fill missing values using imputation.
# 2. Remove samples containing missing values.
#
# In Machine Learning, it is usually better to understand the missing
# values first and then choose an appropriate strategy.

In [ ]:
# View the dataset
car_sales_missing

In [ ]:
# Check the number of doors in the dataset
car_sales_missing["Doors"].value_counts(dropna=False)

# ## 1.2 Option 1 — Fill Missing Data with Pandas
#
# We can fill missing values using Pandas before moving to
# Scikit-Learn preprocessing.

In [ ]:
# Fill missing "Make" values
car_sales_missing["Make"] = car_sales_missing["Make"].fillna("missing")

# Fill missing "Colour" values
car_sales_missing["Colour"] = car_sales_missing["Colour"].fillna("missing")

# Fill missing "Odometer (KM)" values with the mean
car_sales_missing["Odometer (KM)"] = (
    car_sales_missing["Odometer (KM)"]
    .fillna(car_sales_missing["Odometer (KM)"].mean())
)

# Fill missing "Doors" values with 4
car_sales_missing["Doors"] = car_sales_missing["Doors"].fillna(4)

In [ ]:
# Check missing values again
print("Missing values after Pandas imputation:")
print(car_sales_missing.isna().sum())

In [ ]:
# Remove rows where the target "Price" is missing
car_sales_missing = car_sales_missing.dropna(subset=["Price"])

print("Missing values after removing missing Price rows:")
print(car_sales_missing.isna().sum())

In [ ]:
print("Remaining rows:", len(car_sales_missing))

# ## 1.3 Separate Features and Target
#
# X = features/input variables
#
# y = target/output variable
#
# Here, Price is the value we want to predict.

In [ ]:
X = car_sales_missing.drop("Price", axis=1)
y = car_sales_missing["Price"]

print("Features:")
display(X.head())

print("\nTarget:")
display(y.head())

# ## 1.4 Convert Categorical Data into Numbers
#
# Machine Learning models generally require numerical input.
#
# We use:
# - OneHotEncoder for categorical columns
# - ColumnTransformer to apply transformations to selected columns

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ["Make", "Colour", "Doors"]

one_hot = OneHotEncoder(handle_unknown="ignore")

transformer = ColumnTransformer(
    [
        (
            "one_hot",
            one_hot,
            categorical_features
        )
    ],
    remainder="passthrough"
)

transformed_X = transformer.fit_transform(X)

print("Transformed feature matrix shape:", transformed_X.shape)

# ## 1.5 Scikit-Learn Preprocessing
#
# The recommended workflow is:
#
# **Split the data first → fit preprocessing on training data →
# transform both training and test data.**
#
# This helps prevent information from the test set leaking into training.

In [ ]:
# Reload the original dataset so we can demonstrate the
# Scikit-Learn preprocessing workflow from the beginning.
car_sales_missing = pd.read_csv(
    "data/car-sales-extended-missing-data.csv"
)

# Drop rows where the target value is missing
car_sales_missing = car_sales_missing.dropna(subset=["Price"])

In [ ]:
# Split into features and target
X = car_sales_missing.drop("Price", axis=1)
y = car_sales_missing["Price"]

In [ ]:
# Split data into training and testing sets
from sklearn.model_selection import train_test_split

np.random.seed(42)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Check missing values in the feature data
print("Missing values in X:")
print(X.isna().sum())

# ## 1.6 Fill Missing Values with SimpleImputer
#
# Different columns can require different strategies:
#
# - Make → "missing"
# - Colour → "missing"
# - Doors → 4
# - Odometer (KM) → mean
#
# We use ColumnTransformer to apply each strategy to the correct column.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Categorical values
cat_imputer = SimpleImputer(
    strategy="constant",
    fill_value="missing"
)

# Door values
door_imputer = SimpleImputer(
    strategy="constant",
    fill_value=4
)

# Numerical values
num_imputer = SimpleImputer(
    strategy="mean"
)

# Define columns
cat_features = ["Make", "Colour"]
door_feature = ["Doors"]
num_features = ["Odometer (KM)"]

# Create the preprocessing transformer
imputer = ColumnTransformer(
    [
        ("cat_imputer", cat_imputer, cat_features),
        ("door_imputer", door_imputer, door_feature),
        ("num_imputer", num_imputer, num_features)
    ]
)

In [ ]:
# Fit the imputer ONLY on training data
filled_X_train = imputer.fit_transform(X_train)

# Transform test data using the already-fitted imputer
filled_X_test = imputer.transform(X_test)

print("Training data after imputation:")
print(filled_X_train[:5])

# ## 1.7 Convert the Imputed Data Back to DataFrames

In [ ]:
columns = [
    "Make",
    "Colour",
    "Doors",
    "Odometer (KM)"
]

car_sales_filled_train = pd.DataFrame(
    filled_X_train,
    columns=columns
)

car_sales_filled_test = pd.DataFrame(
    filled_X_test,
    columns=columns
)

In [ ]:
print("Missing values in training data:")
print(car_sales_filled_train.isna().sum())

In [ ]:
print("Missing values in test data:")
print(car_sales_filled_test.isna().sum())

# ## 1.8 One-Hot Encode the Features
#
# Categorical columns are converted into numerical columns.

In [ ]:
categorical_features = ["Make", "Colour", "Doors"]

one_hot = OneHotEncoder(
    handle_unknown="ignore"
)

transformer = ColumnTransformer(
    [
        (
            "one_hot",
            one_hot,
            categorical_features
        )
    ],
    remainder="passthrough"
)

# Fit on training data and transform training data
transformed_X_train = transformer.fit_transform(
    car_sales_filled_train
)

# Transform test data using the same fitted transformer
transformed_X_test = transformer.transform(
    car_sales_filled_test
)

print(
    "Transformed training shape:",
    transformed_X_train.shape
)

print(
    "Transformed testing shape:",
    transformed_X_test.shape
)

# ## 1.9 Fit a Machine Learning Model
#
# Now that the missing values have been handled and categorical
# features have been encoded, the data is ready for a model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train the model using transformed training data
model.fit(
    transformed_X_train,
    y_train
)

In [ ]:
# Evaluate on transformed test data
score = model.score(
    transformed_X_test,
    y_test
)

print("Model R² score:", round(score, 3))

# ##  Complete Data Preprocessing Workflow
#
# ```text
# Raw Data
#    ↓
# Inspect Missing Values
#    ↓
# Remove Rows with Missing Target
#    ↓
# Split Features & Target
#    ↓
# Train/Test Split
#    ↓
# Impute Missing Values
#    ↓
# One-Hot Encode Categorical Features
#    ↓
# Train Machine Learning Model
#    ↓
# Evaluate Model
# ```
#
# ##  Key Takeaways
#
# - Missing values must be handled before model training.
# - Features and target should be separated.
# - Train/test splitting should happen before fitting preprocessing.
# - `SimpleImputer` handles missing values.
# - `OneHotEncoder` converts categorical data into numerical features.
# - `ColumnTransformer` applies different transformations to different columns.
# - The same fitted preprocessing steps must be used on test data.
#
# ##  Next Step
#
# Continue with:
#
# **Regression — Linear Regression, Random Forest Regression,
# MAE, MSE, RMSE and R².**